
# 05 - Prepare Dataset Malaysia

Notebook ini digunakan untuk:
1. Membaca dataset hasil anotasi Roboflow.
2. Memeriksa struktur train dan test.
3. Menyamakan ID kelas menjadi Unripe (0) dan Ripe (1).
4. Memvalidasi file anotasi YOLO.
5. Menyiapkan dataset untuk eksperimen cross-domain.

Dataset sumber: Oil Palm Ripeness Malaysia v4
Format anotasi: YOLO


In [ ]:

from pathlib import Path
import pandas as pd

# Direktori utama project
PROJECT_DIR = Path.cwd().parent

# Dataset hasil ekspor Roboflow yang sudah diekstrak
SOURCE_DIR = (
    PROJECT_DIR
    / "datasets"
    / "dataset-malaysia"
    / "roboflow_export"
)

# Direktori hasil preprocessing
OUTPUT_DIR = (
    PROJECT_DIR
    / "datasets"
    / "processed"
    / "dataset-malaysia"
)

print("Project directory:", PROJECT_DIR)
print("Source directory:", SOURCE_DIR)
print("Source ditemukan:", SOURCE_DIR.exists())
print("Train ditemukan:", (SOURCE_DIR / "train").exists())
print("Test ditemukan:", (SOURCE_DIR / "test").exists())
print("data.yaml ditemukan:", (SOURCE_DIR / "data.yaml").exists())
print("Output directory:", OUTPUT_DIR)

In [ ]:
# Membaca konfigurasi dataset Roboflow
yaml_path = SOURCE_DIR / "data.yaml"

print("=== ISI DATA.YAML ===")
print(yaml_path.read_text(encoding="utf-8"))

# Ekstensi gambar yang didukung
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

# Memeriksa jumlah gambar dan label
for split in ["train", "valid", "test"]:
    split_dir = SOURCE_DIR / split

    if not split_dir.exists():
        print(f"\n{split.upper()}: folder tidak ditemukan")
        continue

    image_dir = split_dir / "images"
    label_dir = split_dir / "labels"

    images = (
        [p for p in image_dir.rglob("*")
         if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS]
        if image_dir.exists() else []
    )

    labels = list(label_dir.rglob("*.txt")) if label_dir.exists() else []

    print(f"\n=== {split.upper()} ===")
    print("Jumlah gambar :", len(images))
    print("Jumlah label  :", len(labels))
    print("Folder images :", image_dir.exists())
    print("Folder labels :", label_dir.exists())

In [ ]:
from collections import Counter

class_counts = Counter()
audit_results = []

for split in ["train", "test"]:
    label_dir = SOURCE_DIR / split / "labels"
    image_dir = SOURCE_DIR / split / "images"

    label_files = list(label_dir.rglob("*.txt"))
    image_files = [
        p for p in image_dir.rglob("*")
        if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS
    ]

    image_stems = {p.stem for p in image_files}
    label_stems = {p.stem for p in label_files}

    invalid_class = 0
    invalid_bbox = 0
    empty_labels = 0
    object_count = 0
    counts = Counter()

    for label_file in label_files:
        lines = label_file.read_text(encoding="utf-8").splitlines()

        if not lines:
            empty_labels += 1

        for line in lines:
            parts = line.split()

            if len(parts) != 5:
                invalid_bbox += 1
                continue

            try:
                class_id = int(parts[0])
                x, y, w, h = map(float, parts[1:])
            except ValueError:
                invalid_bbox += 1
                continue

            if class_id not in [0, 1]:
                invalid_class += 1
                continue

            if not (
                0 <= x <= 1 and
                0 <= y <= 1 and
                0 < w <= 1 and
                0 < h <= 1 and
                x - w / 2 >= 0 and
                y - h / 2 >= 0 and
                x + w / 2 <= 1 and
                y + h / 2 <= 1
            ):
                invalid_bbox += 1

            counts[class_id] += 1
            object_count += 1

    class_counts.update(counts)

    audit_results.append({
        "split": split,
        "images": len(image_files),
        "labels": len(label_files),
        "missing_labels": len(image_stems - label_stems),
        "orphan_labels": len(label_stems - image_stems),
        "objects": object_count,
        "empty_labels": empty_labels,
        "invalid_class": invalid_class,
        "invalid_bbox": invalid_bbox,
        "Ripe_ID0": counts[0],
        "Unripe_ID1": counts[1],
    })

audit_df = pd.DataFrame(audit_results)

display(audit_df)

print("\nTOTAL OBJEK")
print("Ripe (ID 0):", class_counts[0])
print("Unripe (ID 1):", class_counts[1])
print("Total:", sum(class_counts.values()))

In [ ]:
import shutil
from collections import Counter

# Mapping kelas Roboflow ke standar penelitian
CLASS_MAPPING = {
    0: 1,  # Ripe -> Ripe
    1: 0,  # Unripe -> Unripe
}

CLASS_NAMES = ["Unripe", "Ripe"]

# Membuat direktori hasil
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

remap_summary = []

for split in ["train", "test"]:
    source_images = SOURCE_DIR / split / "images"
    source_labels = SOURCE_DIR / split / "labels"

    output_images = OUTPUT_DIR / split / "images"
    output_labels = OUTPUT_DIR / split / "labels"

    output_images.mkdir(parents=True, exist_ok=True)
    output_labels.mkdir(parents=True, exist_ok=True)

    image_files = [
        p for p in source_images.rglob("*")
        if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS
    ]

    class_counts = Counter()
    total_objects = 0

    for image_path in image_files:
        # Salin gambar tanpa mengubah isinya
        shutil.copy2(
            image_path,
            output_images / image_path.name
        )

        # Baca file label yang sesuai
        label_path = source_labels / f"{image_path.stem}.txt"

        remapped_lines = []

        for line in label_path.read_text(encoding="utf-8").splitlines():
            parts = line.split()

            old_class_id = int(parts[0])
            new_class_id = CLASS_MAPPING[old_class_id]

            # Ganti ID kelas, koordinat tetap
            parts[0] = str(new_class_id)

            remapped_lines.append(" ".join(parts))

            class_counts[new_class_id] += 1
            total_objects += 1

        # Simpan label hasil remapping
        output_label_path = output_labels / f"{image_path.stem}.txt"

        output_label_path.write_text(
            "\n".join(remapped_lines) + ("\n" if remapped_lines else ""),
            encoding="utf-8"
        )

    remap_summary.append({
        "split": split,
        "images": len(image_files),
        "objects": total_objects,
        "Unripe_ID0": class_counts[0],
        "Ripe_ID1": class_counts[1],
    })

remap_df = pd.DataFrame(remap_summary)

display(remap_df)

print("\nHasil disimpan di:", OUTPUT_DIR)

In [ ]:
# Validasi hasil remapping
validation_results = []

for split in ["train", "test"]:
    image_dir = OUTPUT_DIR / split / "images"
    label_dir = OUTPUT_DIR / split / "labels"

    images = [
        p for p in image_dir.rglob("*")
        if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS
    ]

    labels = list(label_dir.rglob("*.txt"))

    class_counts = Counter()
    invalid_ids = 0
    invalid_boxes = 0

    image_stems = {p.stem for p in images}
    label_stems = {p.stem for p in labels}

    for label_file in labels:
        for line in label_file.read_text(encoding="utf-8").splitlines():
            parts = line.split()

            if len(parts) != 5:
                invalid_boxes += 1
                continue

            try:
                class_id = int(parts[0])
                x, y, w, h = map(float, parts[1:])
            except ValueError:
                invalid_boxes += 1
                continue

            if class_id not in [0, 1]:
                invalid_ids += 1
                continue

            class_counts[class_id] += 1

            if not (
                0 <= x <= 1 and 0 <= y <= 1 and
                0 < w <= 1 and 0 < h <= 1 and
                x - w / 2 >= 0 and y - h / 2 >= 0 and
                x + w / 2 <= 1 and y + h / 2 <= 1
            ):
                invalid_boxes += 1

    validation_results.append({
        "split": split,
        "images": len(images),
        "labels": len(labels),
        "missing_labels": len(image_stems - label_stems),
        "orphan_labels": len(label_stems - image_stems),
        "Unripe_ID0": class_counts[0],
        "Ripe_ID1": class_counts[1],
        "invalid_ids": invalid_ids,
        "invalid_boxes": invalid_boxes,
    })

validation_df = pd.DataFrame(validation_results)
display(validation_df)

In [ ]:
# Membuat konfigurasi dataset Malaysia
yaml_content = """# Dataset Malaysia - Roboflow v4
# Standard class mapping:
# 0 = Unripe
# 1 = Ripe

path: .
train: train/images
test: test/images

nc: 2
names:
  0: Unripe
  1: Ripe
"""

yaml_output = OUTPUT_DIR / "data.yaml"

yaml_output.write_text(yaml_content, encoding="utf-8")

print("Konfigurasi berhasil dibuat:", yaml_output)
print("\nIsi data.yaml:")
print(yaml_output.read_text(encoding="utf-8"))

In [ ]:
# Pemeriksaan lokasi sumber ketiga dataset

DATASET_PATHS = {
    "pabrik_images_train": PROJECT_DIR / "datasets/dataset-pabrik-kalsel/train_rev2/train",
    "pabrik_images_valid": PROJECT_DIR / "datasets/dataset-pabrik-kalsel/valid_rev2/valid",
    "pabrik_images_test": PROJECT_DIR / "datasets/dataset-pabrik-kalsel/test_rev2/test",

    "pabrik_labels_train": PROJECT_DIR / "datasets/processed/dataset-pabrik-kalsel/labels/train",
    "pabrik_labels_valid": PROJECT_DIR / "datasets/processed/dataset-pabrik-kalsel/labels/valid",
    "pabrik_labels_test": PROJECT_DIR / "datasets/processed/dataset-pabrik-kalsel/labels/test",

    "kalteng_images_train": PROJECT_DIR / "datasets/dataset-perkebunan-kalteng/yolov8/train/images",
    "kalteng_images_valid": PROJECT_DIR / "datasets/dataset-perkebunan-kalteng/yolov8/valid/images",
    "kalteng_images_test": PROJECT_DIR / "datasets/dataset-perkebunan-kalteng/yolov8/test/images",

    "kalteng_labels_train": PROJECT_DIR / "datasets/processed/dataset-perkebunan-kalteng/labels/train",
    "kalteng_labels_valid": PROJECT_DIR / "datasets/processed/dataset-perkebunan-kalteng/labels/valid",
    "kalteng_labels_test": PROJECT_DIR / "datasets/processed/dataset-perkebunan-kalteng/labels/test",

    "malaysia_images_train": OUTPUT_DIR / "train/images",
    "malaysia_labels_train": OUTPUT_DIR / "train/labels",
    "malaysia_images_test": OUTPUT_DIR / "test/images",
    "malaysia_labels_test": OUTPUT_DIR / "test/labels",
}

path_check = pd.DataFrame([
    {
        "folder": name,
        "exists": path.exists(),
        "path": str(path)
    }
    for name, path in DATASET_PATHS.items()
])

display(path_check)

print("Semua folder ditemukan:", path_check["exists"].all())

In [ ]:
# Audit keberadaan kelas target dan non-target
# pada anotasi asli Dataset Pabrik dan Kalteng

ORIGINAL_LABEL_PATHS = {
    "Pabrik": {
        "train": PROJECT_DIR / "datasets/dataset-pabrik-kalsel/train_rev2/train",
        "valid": PROJECT_DIR / "datasets/dataset-pabrik-kalsel/valid_rev2/valid",
        "test": PROJECT_DIR / "datasets/dataset-pabrik-kalsel/test_rev2/test",
    },
    "Kalteng": {
        "train": PROJECT_DIR / "datasets/dataset-perkebunan-kalteng/yolov8/train/labels",
        "valid": PROJECT_DIR / "datasets/dataset-perkebunan-kalteng/yolov8/valid/labels",
        "test": PROJECT_DIR / "datasets/dataset-perkebunan-kalteng/yolov8/test/labels",
    }
}

TARGET_CLASSES = {
    "Pabrik": {3, 4},  # TBS masak, TBS mentah
    "Kalteng": {2, 4}, # Ripe, Unripe
}

audit_mixed = []

for dataset, splits in ORIGINAL_LABEL_PATHS.items():
    for split, folder in splits.items():
        label_files = list(folder.glob("*.txt"))

        target_only = 0
        mixed_classes = 0
        non_target_only = 0
        empty = 0

        for file in label_files:
            lines = file.read_text(encoding="utf-8").splitlines()

            if not lines:
                empty += 1
                continue

            class_ids = {
                int(line.split()[0])
                for line in lines
                if len(line.split()) == 5
            }

            has_target = bool(class_ids & TARGET_CLASSES[dataset])
            has_other = bool(class_ids - TARGET_CLASSES[dataset])

            if has_target and has_other:
                mixed_classes += 1
            elif has_target:
                target_only += 1
            else:
                non_target_only += 1

        audit_mixed.append({
            "dataset": dataset,
            "split": split,
            "target_only_images": target_only,
            "mixed_target_non_target": mixed_classes,
            "non_target_only_images": non_target_only,
            "empty_original_labels": empty,
        })

mixed_df = pd.DataFrame(audit_mixed)
display(mixed_df)

In [ ]:
import shutil
from collections import Counter

FILTERED_DIR = PROJECT_DIR / "datasets" / "processed" / "filtered"

# Sumber gambar asli
IMAGE_SOURCES = {
    "Pabrik": {
        "train": DATASET_PATHS["pabrik_images_train"],
        "valid": DATASET_PATHS["pabrik_images_valid"],
        "test": DATASET_PATHS["pabrik_images_test"],
    },
    "Kalteng": {
        "train": DATASET_PATHS["kalteng_images_train"],
        "valid": DATASET_PATHS["kalteng_images_valid"],
        "test": DATASET_PATHS["kalteng_images_test"],
    },
}

# Sumber label asli
LABEL_SOURCES = {
    "Pabrik": ORIGINAL_LABEL_PATHS["Pabrik"],
    "Kalteng": ORIGINAL_LABEL_PATHS["Kalteng"],
}

# Label hasil remapping
PROCESSED_LABELS = {
    "Pabrik": {
        "train": DATASET_PATHS["pabrik_labels_train"],
        "valid": DATASET_PATHS["pabrik_labels_valid"],
        "test": DATASET_PATHS["pabrik_labels_test"],
    },
    "Kalteng": {
        "train": DATASET_PATHS["kalteng_labels_train"],
        "valid": DATASET_PATHS["kalteng_labels_valid"],
        "test": DATASET_PATHS["kalteng_labels_test"],
    },
}

# Kelas target dalam label asli
TARGET_IDS = {
    "Pabrik": {3, 4},
    "Kalteng": {2, 4},
}

filter_summary = []

for dataset in ["Pabrik", "Kalteng"]:
    for split in ["train", "valid", "test"]:

        image_dir = IMAGE_SOURCES[dataset][split]
        original_label_dir = LABEL_SOURCES[dataset][split]
        processed_label_dir = PROCESSED_LABELS[dataset][split]

        output_name = (
            "dataset-pabrik-kalsel"
            if dataset == "Pabrik"
            else "dataset-perkebunan-kalteng"
        )

        output_images = FILTERED_DIR / output_name / split / "images"
        output_labels = FILTERED_DIR / output_name / split / "labels"

        output_images.mkdir(parents=True, exist_ok=True)
        output_labels.mkdir(parents=True, exist_ok=True)

        kept = 0
        skipped_mixed = 0
        skipped_non_target = 0
        missing = 0

        image_files = [
            p for p in image_dir.rglob("*")
            if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS
        ]

        for image_path in image_files:
            original_label = original_label_dir / f"{image_path.stem}.txt"
            processed_label = processed_label_dir / f"{image_path.stem}.txt"

            if not original_label.exists() or not processed_label.exists():
                missing += 1
                continue

            lines = original_label.read_text(
                encoding="utf-8"
            ).splitlines()

            original_ids = {
                int(line.split()[0])
                for line in lines
                if len(line.split()) == 5
            }

            if not original_ids.intersection(TARGET_IDS[dataset]):
                skipped_non_target += 1
                continue

            if not original_ids.issubset(TARGET_IDS[dataset]):
                skipped_mixed += 1
                continue

            shutil.copy2(
                image_path,
                output_images / image_path.name
            )

            shutil.copy2(
                processed_label,
                output_labels / f"{image_path.stem}.txt"
            )

            kept += 1

        filter_summary.append({
            "dataset": dataset,
            "split": split,
            "original_images": len(image_files),
            "kept_target_only": kept,
            "skipped_mixed": skipped_mixed,
            "skipped_non_target": skipped_non_target,
            "missing_files": missing,
        })

filter_df = pd.DataFrame(filter_summary)
display(filter_df)

print("Filtered dataset disimpan di:", FILTERED_DIR)

In [ ]:
# Validasi dataset hasil filtering

filtered_validation = []

for dataset in ["dataset-pabrik-kalsel",
                "dataset-perkebunan-kalteng"]:

    for split in ["train", "valid", "test"]:

        image_dir = FILTERED_DIR / dataset / split / "images"
        label_dir = FILTERED_DIR / dataset / split / "labels"

        images = [
            p for p in image_dir.rglob("*")
            if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS
        ]

        labels = list(label_dir.glob("*.txt"))

        image_stems = {p.stem for p in images}
        label_stems = {p.stem for p in labels}

        counts = Counter()
        invalid_ids = 0
        invalid_boxes = 0

        for label in labels:
            for line in label.read_text(encoding="utf-8").splitlines():
                parts = line.split()

                if len(parts) != 5:
                    invalid_boxes += 1
                    continue

                try:
                    class_id = int(parts[0])
                    x, y, w, h = map(float, parts[1:])
                except ValueError:
                    invalid_boxes += 1
                    continue

                if class_id not in [0, 1]:
                    invalid_ids += 1
                    continue

                counts[class_id] += 1

                if not (
                    0 <= x <= 1 and 0 <= y <= 1 and
                    0 < w <= 1 and 0 < h <= 1 and
                    x - w / 2 >= 0 and y - h / 2 >= 0 and
                    x + w / 2 <= 1 and y + h / 2 <= 1
                ):
                    invalid_boxes += 1

        filtered_validation.append({
            "dataset": dataset,
            "split": split,
            "images": len(images),
            "labels": len(labels),
            "missing_labels": len(image_stems - label_stems),
            "orphan_labels": len(label_stems - image_stems),
            "Unripe_ID0": counts[0],
            "Ripe_ID1": counts[1],
            "invalid_ids": invalid_ids,
            "invalid_boxes": invalid_boxes,
        })

filtered_validation_df = pd.DataFrame(filtered_validation)
display(filtered_validation_df)

print(
    "Semua pemeriksaan lolos:",
    (
        (filtered_validation_df["images"] == filtered_validation_df["labels"]).all()
        and (filtered_validation_df["missing_labels"] == 0).all()
        and (filtered_validation_df["orphan_labels"] == 0).all()
        and (filtered_validation_df["invalid_ids"] == 0).all()
        and (filtered_validation_df["invalid_boxes"] == 0).all()
    )
)

In [ ]:
from collections import Counter
import pandas as pd

EXPERIMENT_DIR = PROJECT_DIR / "datasets" / "experiments"

validation_summary = []

for scenario in [
    "lodo_1_pabrik",
    "lodo_2_kalteng",
    "lodo_3_malaysia"
]:
    for split in ["train", "val", "test"]:

        image_dir = EXPERIMENT_DIR / scenario / split / "images"
        label_dir = EXPERIMENT_DIR / scenario / split / "labels"

        images = [
            p for p in image_dir.rglob("*")
            if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS
        ]

        labels = list(label_dir.glob("*.txt"))

        image_stems = {p.stem for p in images}
        label_stems = {p.stem for p in labels}

        counts = Counter()
        invalid_ids = 0
        invalid_boxes = 0

        for label in labels:
            for line in label.read_text(encoding="utf-8").splitlines():
                parts = line.split()

                if len(parts) != 5:
                    invalid_boxes += 1
                    continue

                try:
                    class_id = int(parts[0])
                    x, y, w, h = map(float, parts[1:])
                except ValueError:
                    invalid_boxes += 1
                    continue

                if class_id not in [0, 1]:
                    invalid_ids += 1
                    continue

                counts[class_id] += 1

                if not (
                    0 <= x <= 1 and 0 <= y <= 1
                    and 0 < w <= 1 and 0 < h <= 1
                    and x - w / 2 >= 0
                    and y - h / 2 >= 0
                    and x + w / 2 <= 1
                    and y + h / 2 <= 1
                ):
                    invalid_boxes += 1

        validation_summary.append({
            "scenario": scenario,
            "split": split,
            "images": len(images),
            "labels": len(labels),
            "missing_labels": len(image_stems - label_stems),
            "orphan_labels": len(label_stems - image_stems),
            "Unripe": counts[0],
            "Ripe": counts[1],
            "invalid_ids": invalid_ids,
            "invalid_boxes": invalid_boxes,
        })

validation_df = pd.DataFrame(validation_summary)

display(validation_df)

print(
    "\nSemua pemeriksaan lolos:",
    (
        (validation_df["images"] == validation_df["labels"]).all()
        and (validation_df["missing_labels"] == 0).all()
        and (validation_df["orphan_labels"] == 0).all()
        and (validation_df["invalid_ids"] == 0).all()
        and (validation_df["invalid_boxes"] == 0).all()
    )
)

In [ ]:
from pathlib import Path
import random
import cv2
import matplotlib.pyplot as plt

# Direktori utama proyek
PROJECT_DIR = Path.cwd().parent

# Lokasi dataset LODO
LODO_DIR = PROJECT_DIR / "datasets" / "experiments"

# Ambil sampel dari setiap domain:
# Pabrik: test LODO 1
# Kalteng: train LODO 1
# Malaysia: train LODO 1
samples = [
    {
        "domain": "Pabrik Kalsel",
        "image_dir": LODO_DIR / "lodo_1_pabrik" / "test" / "images",
        "label_dir": LODO_DIR / "lodo_1_pabrik" / "test" / "labels",
    },
    {
        "domain": "Perkebunan Kalteng",
        "image_dir": LODO_DIR / "lodo_1_pabrik" / "train" / "images",
        "label_dir": LODO_DIR / "lodo_1_pabrik" / "train" / "labels",
        "prefix": "kalteng_",
    },
    {
        "domain": "Malaysia",
        "image_dir": LODO_DIR / "lodo_1_pabrik" / "train" / "images",
        "label_dir": LODO_DIR / "lodo_1_pabrik" / "train" / "labels",
        "prefix": "malaysia_",
    },
]

CLASS_NAMES = {
    0: "Unripe",
    1: "Ripe",
}

# Visualisasi 2 gambar dari setiap domain
fig, axes = plt.subplots(3, 2, figsize=(14, 18))

for row, source in enumerate(samples):
    image_dir = source["image_dir"]
    label_dir = source["label_dir"]
    prefix = source.get("prefix")

    image_paths = sorted(image_dir.glob("*"))

    if prefix:
        image_paths = [
            p for p in image_paths
            if p.stem.startswith(prefix)
        ]

    image_paths = [
        p for p in image_paths
        if p.suffix.lower() in [".jpg", ".jpeg", ".png", ".bmp"]
        and (label_dir / f"{p.stem}.txt").exists()
    ]

    if len(image_paths) < 2:
        raise ValueError(
            f"Sampel tidak cukup untuk domain {source['domain']}"
        )

    selected = random.sample(image_paths, 2)

    for col, image_path in enumerate(selected):
        label_path = label_dir / f"{image_path.stem}.txt"

        image = cv2.imread(str(image_path))
        if image is None:
            raise ValueError(f"Gagal membaca: {image_path}")

        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        height, width = image.shape[:2]

        # Gambar bounding box
        with open(label_path, "r") as f:
            lines = f.readlines()

        for line in lines:
            values = line.strip().split()

            if len(values) != 5:
                continue

            class_id = int(values[0])
            x_center, y_center, box_width, box_height = map(
                float, values[1:]
            )

            x1 = int((x_center - box_width / 2) * width)
            y1 = int((y_center - box_height / 2) * height)
            x2 = int((x_center + box_width / 2) * width)
            y2 = int((y_center + box_height / 2) * height)

            color = (0, 255, 0) if class_id == 0 else (255, 165, 0)

            cv2.rectangle(
                image,
                (x1, y1),
                (x2, y2),
                color,
                2
            )

            label = CLASS_NAMES.get(class_id, str(class_id))

            cv2.putText(
                image,
                label,
                (x1, max(y1 - 8, 15)),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.6,
                color,
                2
            )

        axes[row, col].imshow(image)
        axes[row, col].set_title(
            f"{source['domain']} - Sampel {col + 1}"
        )
        axes[row, col].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
import random
import cv2
import matplotlib.pyplot as plt
from pathlib import Path

PROJECT_DIR = Path.cwd().parent
LODO_DIR = PROJECT_DIR / "datasets" / "experiments" / "lodo_1_pabrik"

sources = [
    {
        "domain": "Pabrik Kalsel",
        "split": "test",
        "prefix": "pabrik_",
    },
    {
        "domain": "Perkebunan Kalteng",
        "split": "train",
        "prefix": "kalteng_",
    },
    {
        "domain": "Malaysia",
        "split": "train",
        "prefix": "malaysia_",
    },
]

random.seed(42)

fig, axes = plt.subplots(3, 10, figsize=(25, 11))

for row, source in enumerate(sources):
    image_dir = LODO_DIR / source["split"] / "images"
    label_dir = LODO_DIR / source["split"] / "labels"

    candidates = [
        p for p in image_dir.iterdir()
        if p.is_file()
        and p.suffix.lower() in [".jpg", ".jpeg", ".png", ".bmp"]
        and p.stem.startswith(source["prefix"])
        and (label_dir / f"{p.stem}.txt").exists()
    ]

    print(f"{source['domain']}: {len(candidates)} gambar tersedia")

    selected = random.sample(candidates, min(10, len(candidates)))

    for col in range(10):
        ax = axes[row, col]
        ax.axis("off")

        if col >= len(selected):
            continue

        image_path = selected[col]
        label_path = label_dir / f"{image_path.stem}.txt"

        image = cv2.imread(str(image_path))
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        h, w = image.shape[:2]

        with open(label_path, "r") as f:
            for line in f:
                values = line.split()
                if len(values) != 5:
                    continue

                class_id = int(values[0])
                xc, yc, bw, bh = map(float, values[1:])

                x1 = int((xc - bw / 2) * w)
                y1 = int((yc - bh / 2) * h)
                x2 = int((xc + bw / 2) * w)
                y2 = int((yc + bh / 2) * h)

                color = (0, 255, 0) if class_id == 0 else (255, 165, 0)

                cv2.rectangle(image, (x1, y1), (x2, y2), color, 2)

                name = "Unripe" if class_id == 0 else "Ripe"
                cv2.putText(
                    image, name,
                    (x1, max(y1 - 5, 15)),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.45, color, 1
                )

        ax.imshow(image)
        ax.set_title(f"Sampel {col + 1}", fontsize=9)

    axes[row, 0].text(
        -0.15, 0.5, source["domain"],
        transform=axes[row, 0].transAxes,
        rotation=90, va="center", ha="center",
        fontsize=11
    )

plt.tight_layout()
plt.show()

In [ ]:
# Tampilkan nama file sampel yang dipilih
# Gunakan seed dan urutan yang sama seperti visualisasi sebelumnya.

import random
from pathlib import Path

PROJECT_DIR = Path.cwd().parent
LODO_DIR = PROJECT_DIR / "datasets" / "experiments" / "lodo_1_pabrik"

sources = [
    ("Pabrik Kalsel", "test", "pabrik_"),
    ("Perkebunan Kalteng", "train", "kalteng_"),
    ("Malaysia", "train", "malaysia_"),
]

random.seed(42)

for domain, split, prefix in sources:
    image_dir = LODO_DIR / split / "images"
    label_dir = LODO_DIR / split / "labels"

    candidates = sorted([
        p for p in image_dir.iterdir()
        if p.is_file()
        and p.suffix.lower() in [".jpg", ".jpeg", ".png", ".bmp"]
        and p.stem.startswith(prefix)
        and (label_dir / f"{p.stem}.txt").exists()
    ])

    selected = random.sample(candidates, min(10, len(candidates)))

    print(f"\n{domain}")
    for i, path in enumerate(selected, 1):
        print(f"Sampel {i}: {path.name}")

In [ ]:
from pathlib import Path
import cv2
import matplotlib.pyplot as plt

PROJECT_DIR = Path.cwd().parent

IMAGE_DIR = (
    PROJECT_DIR / "datasets" / "experiments" /
    "lodo_1_pabrik" / "train" / "images"
)

LABEL_DIR = (
    PROJECT_DIR / "datasets" / "experiments" /
    "lodo_1_pabrik" / "train" / "labels"
)

filenames = [
    "kalteng_frame_6926_png.rf.151b7a11738ea3fa5ec69184f0998808.jpg",
    "malaysia_TrainUnripe-42-_jpg.rf.244ad7c8e80b12f3fdd54ca99bf86e12.jpg",
]

fig, axes = plt.subplots(1, 2, figsize=(18, 9))

for ax, filename in zip(axes, filenames):
    image_path = IMAGE_DIR / filename
    label_path = LABEL_DIR / f"{Path(filename).stem}.txt"

    if not image_path.exists():
        raise FileNotFoundError(image_path)

    if not label_path.exists():
        raise FileNotFoundError(label_path)

    image = cv2.imread(str(image_path))
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    h, w = image.shape[:2]

    with open(label_path, "r") as f:
        lines = f.readlines()

    for line in lines:
        values = line.strip().split()
        if len(values) != 5:
            continue

        class_id = int(values[0])
        xc, yc, bw, bh = map(float, values[1:])

        x1 = int((xc - bw / 2) * w)
        y1 = int((yc - bh / 2) * h)
        x2 = int((xc + bw / 2) * w)
        y2 = int((yc + bh / 2) * h)

        color = (0, 255, 0) if class_id == 0 else (255, 165, 0)

        cv2.rectangle(image, (x1, y1), (x2, y2), color, 3)

        class_name = "Unripe" if class_id == 0 else "Ripe"

        cv2.putText(
            image,
            class_name,
            (x1, max(y1 - 10, 20)),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.8,
            color,
            2
        )

    ax.imshow(image)
    ax.set_title(filename, fontsize=9)
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
from pathlib import Path
import pandas as pd

PROJECT_DIR = Path.cwd().parent

LABEL_DIR = (
    PROJECT_DIR / "datasets" / "experiments"
    / "lodo_1_pabrik" / "train" / "labels"
)

rows = []

for label_path in LABEL_DIR.glob("malaysia_*.txt"):
    with open(label_path, "r") as f:
        lines = [
            line.strip()
            for line in f
            if line.strip()
        ]

    rows.append({
        "filename": label_path.name,
        "jumlah_bbox": len(lines),
        "jumlah_unripe": sum(
            line.split()[0] == "0" for line in lines
        ),
        "jumlah_ripe": sum(
            line.split()[0] == "1" for line in lines
        )
    })

audit_df = pd.DataFrame(rows)

print("Total gambar Malaysia:", len(audit_df))
print("\nDistribusi jumlah bounding box:")
print(audit_df["jumlah_bbox"].value_counts().sort_index())

print("\nGambar dengan lebih dari 1 bounding box:")
display(
    audit_df[audit_df["jumlah_bbox"] > 1]
    .sort_values("jumlah_bbox", ascending=False)
    .head(20)
)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import cv2
from pathlib import Path

PROJECT_DIR = Path.cwd().parent

BASE_DIR = (
    PROJECT_DIR / "datasets" / "experiments"
    / "lodo_1_pabrik" / "train"
)

IMAGE_DIR = BASE_DIR / "images"
LABEL_DIR = BASE_DIR / "labels"

# Hitung jumlah bounding box setiap gambar Malaysia
rows = []

for label_path in LABEL_DIR.glob("malaysia_*.txt"):
    with open(label_path, "r") as f:
        lines = [line.strip() for line in f if line.strip()]

    rows.append({
        "filename": label_path.stem,
        "jumlah_bbox": len(lines)
    })

audit_df = pd.DataFrame(rows)

# Fokus pada gambar dengan minimal 8 bounding box
outliers = (
    audit_df[audit_df["jumlah_bbox"] >= 8]
    .sort_values("jumlah_bbox", ascending=False)
    .reset_index(drop=True)
)

print("Jumlah gambar yang perlu diperiksa:", len(outliers))
display(outliers)

# Visualisasikan maksimal 12 gambar dengan bbox terbanyak
selected = outliers.head(12)

fig, axes = plt.subplots(3, 4, figsize=(18, 13))
axes = axes.flatten()

for ax, (_, row) in zip(axes, selected.iterrows()):
    stem = row["filename"]

    image_path = next(
        (
            p for p in IMAGE_DIR.glob(f"{stem}.*")
            if p.suffix.lower() in [".jpg", ".jpeg", ".png", ".bmp"]
        ),
        None
    )

    label_path = LABEL_DIR / f"{stem}.txt"

    if image_path is None:
        ax.set_title(f"Gambar tidak ditemukan: {stem[:15]}")
        ax.axis("off")
        continue

    image = cv2.imread(str(image_path))
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    h, w = image.shape[:2]

    with open(label_path, "r") as f:
        for line in f:
            values = line.split()
            if len(values) != 5:
                continue

            class_id = int(values[0])
            xc, yc, bw, bh = map(float, values[1:])

            x1 = int((xc - bw / 2) * w)
            y1 = int((yc - bh / 2) * h)
            x2 = int((xc + bw / 2) * w)
            y2 = int((yc + bh / 2) * h)

            color = (0, 255, 0) if class_id == 0 else (255, 165, 0)

            cv2.rectangle(image, (x1, y1), (x2, y2), color, 2)

    ax.imshow(image)
    ax.set_title(f"{row['jumlah_bbox']} bounding box", fontsize=10)
    ax.axis("off")

for ax in axes[len(selected):]:
    ax.axis("off")

plt.tight_layout()
plt.show()